In [5]:
!pip install python-dotenv pyodbc

   ---------------------------------------- 0.0/71.0 kB ? eta -:--:--
   ----- ---------------------------------- 10.2/71.0 kB ? eta -:--:--
   ----------- ---------------------------- 20.5/71.0 kB 131.3 kB/s eta 0:00:01
   ---------------------------------- ----- 61.4/71.0 kB 409.6 kB/s eta 0:00:01
   ---------------------------------------- 71.0/71.0 kB 430.6 kB/s eta 0:00:00


In [2]:
%%writefile .env
DB_SERVER=DESKTOP-BO672N2\SQLEXPRESS
DB_DATABASE=PlaneCrashDB1

Overwriting .env


In [1]:
from dotenv import load_dotenv
import os
import pyodbc

# Load .env file
load_dotenv(override=True)

# Get variables
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')

print(database)

# Build connection string
conn_str = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'   # Or add UID/PWD if not using Trusted
)

#Try connecting
try:
    conn = pyodbc.connect(conn_str)
    print("Connection successful!")
except Exception as e:
    print("Connection failed:", e)


PlaneCrashDB1
Connection successful!


In [2]:
import pandas as pd

In [3]:
query = "select * from crashes;"
df = pd.read_sql(query, conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_36444\3474020330.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,ID,Date,Time,Location,Operator,Flight,Route,Type,Registration,CN_IN,Aboard,Fatalities,Ground,Survivors,SurvivalRate,Summary,ClustID,decade
0,1,1933-04-04,12:30,"OFF BARNEGAT, NEW JERSEY",MILITARY - U.S. NAVY,Not Available,NOT AVAILABLE,GOODYEAR-ZEPPELIN U.S.S. AKRON (AIRSHIP),ZRS-4,Not Available,76,73,0,3,0.039,"While cruising at 1,600 feet off New Jersey, s...",High Fatality,1930
1,2,1950-03-12,14:50,"LLANDOW AIRPORT, CARDIFF, WALES",FAIRFLIGHT LTD.,Not Available,LLANDOW - DUBLIN,AVRO 689 TUDOR 5,G-AKBY,1417,83,80,0,3,0.036,During the approach to Runway 28 at Llandow Ai...,High Fatality,1950
2,3,1952-03-26,Not Available,"MOSCOW, RUSSIA",AEROFLOT,Not Available,NOT AVAILABLE,NOT AVAILABLE,Not Available,Not Available,70,70,0,0,0.000,The plane overshot the runway and collided wit...,High Fatality,1950
3,4,1952-12-20,6:30,"MOSES LAKE, WASHINGTON",MILITARY - U.S. AIR FORCE,Not Available,NOT AVAILABLE,DOUGLAS C-124A GLOBEMASTER,50-100,43238,115,87,0,28,0.243,Within two minutes after takeoff the aircraft ...,High Fatality,1950
4,5,1953-06-18,16:34,"TACHIKAWA AFB, TOKYO, JAPAN",MILITARY - U.S. AIR FORCE,Not Available,TACHIKAWA AB - KIMPO AB,DOUGLAS C-124A GLOBEMASTER II,51-137A,43471,129,129,0,0,0.000,Crashed shortly after taking off from Tachikaw...,High Fatality,1950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
451,452,2008-09-14,3:15,"PERM, RUSSIA",AEROFLOT,821,MOSCOW - PERM,BOEING B-737-505,VP-BKO,25792/2353,88,88,0,0,0.000,The aircraft crashed into a ravine adjacent to...,High Fatality,2000
452,453,2009-01-15,15:06,"NEW YORK, NEW YORK",US AIRWAYS,1549,"NEW YORK, NY- CHARLOTTE, NC",AIRBUS A320-214,N106US,1044,155,0,0,155,1.000,The plane was taking off from La Guardia Airpo...,Low Fatality,2000
453,454,2009-02-25,10:31,"AMSTERDAM, NETHERLANDS",TURKISH AIRLINES,1951,"ISTANBUL, TURKEY - AMSTERDAM, NETHERLANDS",BOEING 737-8F2,TC-JGE,29789/1065,134,9,0,125,0.933,The plane was on final approach to Runway 18R ...,Low Fatality,2000
454,455,2009-05-20,6:30,"NEAR MADIUN, INDONESIA",MILITARY - INDONESIAN AIR FORCE,Not Available,JAKARTA - MADUIN,LOCKHEED C-130 HERCULES,A-1325,1982,112,98,2,14,0.125,"While on approach, the military transport cras...",High Fatality,2000


### Which crash had the highest ground casualties?

In [26]:
highest_ground = df.loc[df["Ground"].idxmax()]
print("Crash with highest ground casualties:\n", highest_ground)

Crash with highest ground casualties:
 ID                                                            403
Date                                                   2001-09-11
Time                                                         8:47
Location                                  NEW YORK CITY, NEW YORK
Operator                                        AMERICAN AIRLINES
Flight                                                         11
Route                                        BOSTON - LOS ANGELES
Type                                             BOEING 767-223ER
Registration                                               N334AA
CN_IN                                                   22332/169
Aboard                                                         92
Fatalities                                                     92
Ground                                                       2750
Survivors                                                       0
SurvivalRate                         

### What is the distribution of crashes by Operator?

In [27]:
import pandas as pd

# Filter out 'Not Available' operators
filtered_df = df[df["Operator"] != "Not Available"]

# Count crashes per operator
crash_counts = filtered_df["Operator"].value_counts()

print(crash_counts)


Operator
AEROFLOT                       49
AIR FRANCE                     11
PAN AMERICAN WORLD AIRWAYS     11
MILITARY - U.S. AIR FORCE      10
TRANS WORLD AIRLINES            9
                               ..
TACA INTERNATIONAL AIRLINES     1
SPANAIR                         1
ITEK AIR                        1
US AIRWAYS                      1
TURKISH AIRLINES                1
Name: count, Length: 250, dtype: int64


### Which aircraft Type appears most often in the dataset? 

In [28]:
most_common_type = df["Type"].value_counts()
max_count = most_common_type.max()

# Filter only those with frequency equal to the maximum
most_frequent_types = most_common_type[most_common_type == max_count]

print("Most frequent aircraft type(s):")
print(most_frequent_types)


Most frequent aircraft type(s):
Type
TUPOLEV TU-134A              8
MCDONNELL DOUGLAS MD-82      8
MCDONNELL DOUGLAS DC-9-32    8
TUPOLEV TU-154M              8
Name: count, dtype: int64


### Create a new column Decade (from Date) and find the decade with the highest total fatalities.

In [5]:
import pandas as pd

# 1. Create decade column
df["decade"] = pd.to_datetime(df["Date"]).dt.year // 10 * 10

# 2. Group by decade and sum fatalities
decade_fatalities = df.groupby("decade")["Fatalities"].sum().reset_index(name="highest_total_fatality")

# 3. Find max fatalities
max_fatalities = decade_fatalities["highest_total_fatality"].max()

# 4. Filter decades with max fatalities (handles "WITH TIES")
result = decade_fatalities[decade_fatalities["highest_total_fatality"] == max_fatalities]

print(result)


   decade  highest_total_fatality
3    1970                   10783


### Find the crash with the lowest SurvivalRate (excluding 0).

In [30]:
filtered_df = df[df["SurvivalRate"] > 0]
min_rate = filtered_df["SurvivalRate"].min()
lowest_survival = filtered_df[filtered_df["SurvivalRate"] == min_rate]

cols_to_show = ["ID","Date", "Location", "Operator", "Type", "Fatalities", "Survivors", "SurvivalRate"]


print(lowest_survival[cols_to_show].to_string(index=False))

 ID       Date            Location                                 Operator                              Type  Fatalities  Survivors  SurvivalRate
 99 1971-07-30 NEAR MORIOKO, JAPAN ALL NIPPON AIRWAYS /  JAPANESE AIR FORCE BOEING B-727-281 / AIR FORCE F86F         163          1         0.006
268 1987-08-16   ROMULUS, MICHIGAN                       NORTHWEST AIRLINES           MCDONNELL DOUGLAS MD-82         154          1         0.006


### What is the median number of Aboard across all crashes?

In [31]:
median_aboard = df["Aboard"].median()
print("Median Aboard:", median_aboard)


Median Aboard: 116.0


### Get summary statistics (mean, min, max) of SurvivalRate.

In [32]:
survival_stats = df["SurvivalRate"].agg(["mean", "min", "max"])
print("SurvivalRate stats:\n", survival_stats)

SurvivalRate stats:
 mean    0.266664
min     0.000000
max     1.000000
Name: SurvivalRate, dtype: float64


### From the DataFrame, select the first 10 crashes where Fatalities exceeded 90% of Aboard.


In [33]:
# Filter rows where Fatalities > 90% of Aboard
filtered_df = df[df["Fatalities"] > df["Aboard"] * 0.9]

cols_to_show = ["ID","Date", "Location", "Operator", "Type", "Fatalities", "Survivors", "SurvivalRate"]

# Select first 10 rows
top10 = filtered_df[cols_to_show].head(10)

print(top10)

    ID        Date                                           Location  \
0    1  1933-04-04                           OFF BARNEGAT, NEW JERSEY   
1    2  1950-03-12                    LLANDOW AIRPORT, CARDIFF, WALES   
2    3  1952-03-26                                     MOSCOW, RUSSIA   
4    5  1953-06-18                        TACHIKAWA AFB, TOKYO, JAPAN   
5    6  1956-06-20                            ASBURY PARK, NEW JERSEY   
6    7  1956-06-30                              GRAND CANYON, ARIZONA   
8    9  1957-08-11                      NEAR ISSOUDUN, QUEBEC, CANADA   
9   10  1958-08-14  NORTH ATLANTIOCEAN, 100 MILES W OF GALWAY BAY,...   
10  11  1958-10-17                                NEAR KANASH, RUSSIA   
13  14  1961-05-10                                      STAH, ALGERIA   

                                   Operator  \
0                      MILITARY - U.S. NAVY   
1                           FAIRFLIGHT LTD.   
2                                  AEROFLOT   
4       

### Create a Series of total crashes per Operator using indexing and display the top 5 operators with the highest counts.

In [34]:
# Create a Series of total crashes per Operator
crash_counts = df["Operator"].value_counts()

# Display the top 5 operators with the highest counts
top5_operators = crash_counts.head(5)

print("Top 5 Operators with most crashes:")
print(top5_operators)

Top 5 Operators with most crashes:
Operator
AEROFLOT                      49
AIR FRANCE                    11
PAN AMERICAN WORLD AIRWAYS    11
MILITARY - U.S. AIR FORCE     10
TRANS WORLD AIRLINES           9
Name: count, dtype: int64


In [5]:
df.to_csv("output.csv", index=False)